# NLLB-200 for Italian → Piedmontese

Trains the four model conditions and translates the evaluation set with each of them.

| condition | starts from | training |
|---|---|---|
| `NLLB-avg` | NLLB-200-distilled-600M + new `pms_Latn` token | none |
| `NLLB-avg+MT` | `NLLB-avg` | MT on the mined pairs |
| `NLLB-avg+DAE-50K+MT` | `NLLB-avg` | DAE on 50K Wikipedia sentences, then MT |
| `NLLB-avg+DAE-full+MT` | `NLLB-avg` | DAE on all 191,884 sentences, then MT |

The CLiC-it paper has the first three; there `NLLB-avg+DAE-50K+MT` is called `NLLB-avg+DAE+MT`.

Training data comes from the Hugging Face dataset [offelia39/pms-wiki-mined-corpora](https://huggingface.co/datasets/offelia39/pms-wiki-mined-corpora), the evaluation set from [ufal/CrowdsourcingPiedmontese](https://huggingface.co/datasets/ufal/CrowdsourcingPiedmontese). The trained models are on the Hub too (links in §5), so to get the translations without training, run §0, §1 and §5.

Output: `translations/translations.csv`, which the metrics notebook reads. Needs a GPU; the paper models were trained on an A100.

## §0 Setup

Models and outputs are saved to `BASE` on Drive. `HF_HOME` keeps the Hugging Face cache on Drive too, so NLLB isn't downloaded again every session.

In [ ]:
!pip install -q transformers datasets sentencepiece protobuf accelerate

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/Tesi_piedmontese/"
os.environ["HF_HOME"] = BASE + "hf_cache"   # set before importing transformers

In [ ]:
import gc
import json
import random
import re
import shutil

import numpy as np
import pandas as pd
import torch
import datasets
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq,
)
from transformers.trainer_utils import get_last_checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
NLLB    = "facebook/nllb-200-distilled-600M"
HF_DATA = "offelia39/pms-wiki-mined-corpora"

MODELS_DIR = BASE + "models/"
OUT_DIR    = BASE + "translations/"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

STEP1_DIR   = MODELS_DIR + "nllb-600M-pms-avg/"
DAE50K_DIR  = MODELS_DIR + "nllb-600M-pms-avg-dae50k/"
DAEFULL_DIR = MODELS_DIR + "nllb-600M-pms-avg-daefull/"

# condition: model it is fine-tuned from (None = no fine-tuning), and where it is saved
CONDITIONS = {
    "NLLB-avg":             {"init": None,        "dir": STEP1_DIR},
    "NLLB-avg+MT":          {"init": STEP1_DIR,   "dir": MODELS_DIR + "nllb-600M-pms-avg-mt/"},
    "NLLB-avg+DAE-50K+MT":  {"init": DAE50K_DIR,  "dir": MODELS_DIR + "nllb-600M-pms-avg-dae50k-mt/"},
    "NLLB-avg+DAE-full+MT": {"init": DAEFULL_DIR, "dir": MODELS_DIR + "nllb-600M-pms-avg-daefull-mt/"},
}

# while the Hugging Face repos are private, log in first:
# from huggingface_hub import login; login()

## §1 Data

**Wikipedia corpus** (for DAE). The documents of `pmswiki_noleak.jsonl` are shuffled with seed 1, which fixes the order sentences are drawn in, and joined into one line each. DAE-50K takes the first 50K sentences, DAE-full all of them.

**Mined pairs** (for MT). 5% is held out as validation for fine-tuning. The released models were trained on the original `wiki_mined.csv` (5,304 pairs, output of the Wiki miner notebook). The Hub has the cleaned version, `wiki_mined_clean.csv` (4,714 pairs); switch `MINED_CSV` to the Drive file to train on the original.

**Evaluation set.** The `mt` subset of CrowdsourcingPiedmontese: FLORES+ dev and devtest sentences translated by Piedmontese speakers, each with reviewer scores. A few sentences have two translations; the better-rated one is kept, which leaves 140 sentences.

In [ ]:
wiki_path = hf_hub_download(HF_DATA, "pmswiki_noleak.jsonl", repo_type="dataset")
with open(wiki_path, encoding="utf-8") as f:
    docs = [json.loads(line) for line in f]

random.seed(1)
random.shuffle(docs)

wiki_docs = [d["text"].replace("\n", " ").strip() for d in docs]
wiki_docs = [t for t in wiki_docs if t]
print(f"Wikipedia: {len(wiki_docs):,} documents")

In [ ]:
MINED_CSV = hf_hub_download(HF_DATA, "wiki_mined_clean.csv", repo_type="dataset")
# MINED_CSV = BASE + "wiki_mined/wiki_mined.csv"   # original corpus of the released models

mined_df = pd.read_csv(MINED_CSV)
mt_split = (Dataset.from_pandas(mined_df[["source", "target"]].dropna().reset_index(drop=True))
            .train_test_split(test_size=0.05, seed=42))
print(f"Mined pairs: {len(mined_df):,}  ({len(mt_split['train']):,} train, {len(mt_split['test']):,} validation)")

In [ ]:
def mean_score(scores):
    return float(np.mean(scores)) if scores else float("nan")

mt_ds = datasets.load_dataset("ufal/CrowdsourcingPiedmontese", "mt")
rows = [{"flores_id":    ex["flores_id"],
         "source":       ex["ita_Latn"],
         "reference":    ex["pms_Latn"],
         "review_score": mean_score(ex["review_scores"])}
        for split in ("dev", "devtest") for ex in mt_ds[split]]

eval_df = (pd.DataFrame(rows)
           .sort_values("review_score", ascending=False, na_position="last")
           .drop_duplicates(subset="flores_id")
           .reset_index(drop=True))
print(f"Evaluation set: {len(eval_df)} sentences")

overlap = set(mined_df["source"]) & set(eval_df["source"])
assert not overlap, f"{len(overlap)} evaluation sentences are in the training data"

## §2 NLLB-avg
out: `models/nllb-600M-pms-avg/`

*Can be skipped if saved on Drive* (the cell checks).

NLLB has no Piedmontese language token, so `pms_Latn` is added to the tokenizer and its embedding is set to the average of four related languages: Italian, Occitan, Ligurian and Lombard. This model is the zero-shot condition and the starting point for all others.

In [ ]:
if os.path.exists(STEP1_DIR + "config.json"):
    print("Already on Drive:", STEP1_DIR)
else:
    tok = AutoTokenizer.from_pretrained(NLLB)
    tok.add_special_tokens({"additional_special_tokens": ["pms_Latn"]})
    pms_id = tok.convert_tokens_to_ids("pms_Latn")

    # ita_Latn, oci_Latn, lij_Latn, lmo_Latn; hardcoded because
    # convert_tokens_to_ids is unreliable for existing language codes in the fast tokenizer
    donor_ids = [256077, 256134, 256102, 256106]

    model = AutoModelForSeq2SeqLM.from_pretrained(NLLB)
    model.resize_token_embeddings(len(tok))
    emb_in, emb_out = model.get_input_embeddings(), model.get_output_embeddings()
    with torch.no_grad():
        emb_in.weight[pms_id] = emb_in.weight[donor_ids].mean(0)
        if emb_out is not None and emb_out is not emb_in:
            emb_out.weight[pms_id] = emb_out.weight[donor_ids].mean(0)

    tok.save_pretrained(STEP1_DIR)
    model.save_pretrained(STEP1_DIR)
    del tok, model
    gc.collect(); torch.cuda.empty_cache()

# check: vocab 256205, pms_Latn = 256204, and source ids end in </s> + language code
tok = AutoTokenizer.from_pretrained(STEP1_DIR)
tok.src_lang = "ita_Latn"
ids = tok("Ciao mondo")["input_ids"]
print(len(tok), tok.convert_tokens_to_ids("pms_Latn"), ids[-2] == tok.eos_token_id)
del tok

## §3 Denoising pretraining (DAE)
out: `models/nllb-600M-pms-avg-dae50k/`, `models/nllb-600M-pms-avg-daefull/`

*Can be skipped if saved on Drive* (each run checks).

The model learns to rebuild a Piedmontese sentence from a damaged copy: 10% of the words are deleted and 15% replaced by `___`. Source and target are both `pms_Latn`. Documents are split into sentences on `.`, `!`, `?`, and sentences under 5 words are dropped. 5% of the sentences are held out for validation, and the epoch with the lowest validation loss is kept.

The full run takes several hours, so checkpoints go to Drive after each epoch; if Colab disconnects, run the cell again and it continues. The paper's 50K run had no fixed seed, so its sentence sample can't be reproduced exactly; here both runs use seed 42.

In [ ]:
def train_args(out_dir, epochs, lr, warmup):
    return Seq2SeqTrainingArguments(
        output_dir=out_dir, num_train_epochs=epochs,
        per_device_train_batch_size=8, gradient_accumulation_steps=4,
        learning_rate=lr, warmup_steps=warmup, fp16=True,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        logging_steps=50, report_to="none",
    )

def add_noise(text, delete_prob=0.10, mask_prob=0.15):
    words = text.split()
    out = []
    for w in words:
        r = random.random()
        if r < delete_prob:
            continue
        out.append("___" if r < delete_prob + mask_prob else w)
    return " ".join(out) if out else (words[0] if words else text)

In [ ]:
def run_dae(texts, save_dir, max_sentences=None, seed=42):
    if os.path.exists(save_dir + "config.json"):
        print("Already on Drive:", save_dir)
        return
    random.seed(seed)
    ckpt_dir = save_dir.rstrip("/") + "_ckpt/"

    tok = AutoTokenizer.from_pretrained(STEP1_DIR)
    model = AutoModelForSeq2SeqLM.from_pretrained(STEP1_DIR).to(device)
    pms_id, eos_id = tok.convert_tokens_to_ids("pms_Latn"), tok.eos_token_id

    sents = []
    for text in texts:
        parts = re.split(r'(?<=[.!?])\s+', text.strip())
        sents.extend(p.strip() for p in parts if len(p.split()) >= 5)
        if max_sentences and len(sents) >= max_sentences:
            break
    random.shuffle(sents)
    sents = sents[:max_sentences]
    print(f"DAE sentences: {len(sents):,}")

    def tokenize(batch):
        tok.src_lang = "ita_Latn"
        noisy = [add_noise(t) for t in batch["text"]]
        enc = tok(noisy, max_length=128, truncation=True)
        enc["input_ids"] = [ids[:-1] + [pms_id] for ids in enc["input_ids"]]   # source language -> pms_Latn
        tgt = tok(batch["text"], max_length=127, truncation=True)
        enc["labels"] = [[pms_id] + ids[:-2] + [eos_id] for ids in tgt["input_ids"]]
        return enc

    data = (Dataset.from_dict({"text": sents})
            .train_test_split(test_size=0.05, seed=42)
            .map(tokenize, batched=True, remove_columns=["text"]))

    trainer = Seq2SeqTrainer(
        model=model, args=train_args(ckpt_dir, epochs=2, lr=2e-5, warmup=200),
        train_dataset=data["train"], eval_dataset=data["test"],
        processing_class=tok, data_collator=DataCollatorForSeq2Seq(tok, model=model, padding=True),
    )
    last = get_last_checkpoint(ckpt_dir) if os.path.isdir(ckpt_dir) else None
    trainer.train(resume_from_checkpoint=last)

    model.save_pretrained(save_dir)
    tok.save_pretrained(save_dir)
    shutil.rmtree(ckpt_dir, ignore_errors=True)
    del model, tok, trainer
    gc.collect(); torch.cuda.empty_cache()
    print("Saved:", save_dir)

In [ ]:
run_dae(wiki_docs, DAE50K_DIR, max_sentences=50_000)

In [ ]:
run_dae(wiki_docs, DAEFULL_DIR)

## §4 MT fine-tuning
out: `models/nllb-600M-pms-avg-mt/`, `-avg-dae50k-mt/`, `-avg-daefull-mt/`

*Can be skipped if saved on Drive* (each run checks).

The three fine-tuned conditions get the same training: 3 epochs on the mined pairs, Italian (`ita_Latn`) to Piedmontese (`pms_Latn`), keeping the epoch with the lowest validation loss. They only differ in the model they start from.

In [ ]:
def finetune_mt(init_dir, save_dir):
    if os.path.exists(save_dir + "config.json"):
        print("Already on Drive:", save_dir)
        return
    tok = AutoTokenizer.from_pretrained(init_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(init_dir).to(device)
    pms_id, eos_id = tok.convert_tokens_to_ids("pms_Latn"), tok.eos_token_id

    def tokenize(batch):
        tok.src_lang = "ita_Latn"
        enc = tok(batch["source"], max_length=128, truncation=True)
        tgt = tok(batch["target"], max_length=127, truncation=True)
        enc["labels"] = [[pms_id] + ids[:-2] + [eos_id] for ids in tgt["input_ids"]]
        return enc

    data = mt_split.map(tokenize, batched=True, remove_columns=["source", "target"])
    trainer = Seq2SeqTrainer(
        model=model, args=train_args("/tmp/nllb_mt_ckpt", epochs=3, lr=5e-5, warmup=38),
        train_dataset=data["train"], eval_dataset=data["test"],
        processing_class=tok, data_collator=DataCollatorForSeq2Seq(tok, model=model, padding=True),
    )
    trainer.train()

    model.save_pretrained(save_dir)
    tok.save_pretrained(save_dir)
    del model, tok, trainer
    gc.collect(); torch.cuda.empty_cache()
    print("Saved:", save_dir)

In [ ]:
for name, cond in CONDITIONS.items():
    if cond["init"] is None:
        continue
    print(f"── {name} ──")
    finetune_mt(cond["init"], cond["dir"])

## §5 Translation
out: `translations/translations.csv`

Beam search with 4 beams, up to 128 new tokens, decoding forced to start with `pms_Latn`. With `FROM_HUB = True` the released models are used instead of the ones on Drive:
[NLLB-avg](https://huggingface.co/offelia39/nllb-600M-pms-avg) ·
[NLLB-avg+MT](https://huggingface.co/offelia39/nllb-600M-pms-avg-mt) ·
[NLLB-avg+DAE-50K+MT](https://huggingface.co/offelia39/nllb-600M-pms-avg-dae50k-mt) ·
[NLLB-avg+DAE-full+MT](https://huggingface.co/offelia39/nllb-600M-pms-avg-daefull-mt)

In [ ]:
FROM_HUB = False

def translate(texts, tok, model, batch_size=16):
    tok.src_lang = "ita_Latn"
    pms_id = tok.convert_tokens_to_ids("pms_Latn")
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], return_tensors="pt",
                  padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=pms_id,
                                 num_beams=4, max_new_tokens=128)
        out.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return out

In [ ]:
translations = eval_df[["flores_id", "source", "reference"]].copy()

for name, cond in CONDITIONS.items():
    path = "offelia39/" + cond["dir"].rstrip("/").split("/")[-1] if FROM_HUB else cond["dir"]
    print(f"{name}  <-  {path}")
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSeq2SeqLM.from_pretrained(path).to(device).eval()
    translations[name] = translate(translations["source"].tolist(), tok, model)
    del model, tok
    gc.collect(); torch.cuda.empty_cache()

translations.to_csv(OUT_DIR + "translations.csv", index=False)
translations.head()